# 실험 리뷰

`runs/` 산출물을 읽어 오답 taxonomy와 유형별 Accuracy를 확인한다.
무거운 계산은 여기서 하지 않는다 — `python -m src.run` 으로 돌리고, 결과만 본다.

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
from src.evaluate import load_predictions, summarize, disagreement, select_retry_targets
from src.compare import collect, to_markdown

RUNS = Path('../runs')
sorted(p.name for p in RUNS.iterdir() if p.is_dir())

In [ ]:
run_id = ''  # 위 목록에서 고른다
preds = load_predictions(RUNS / run_id / 'predictions.jsonl')
summarize(preds)

In [ ]:
# 오답만 모아 직접 본다 — 자동 지표보다 이쪽이 다음 실험을 정한다
wrong = [p for p in preds if p.correct is False]
print(len(wrong), '오답')
for p in wrong[:20]:
    print(p.id, '| pred', p.pred, '| gold', p.gold, '| conf', p.confidence, '|', p.qtype)

In [ ]:
# 모델 간 불일치 → 선택적 고해상도 재추론 대상
runs = {r: load_predictions(RUNS / r / 'predictions.jsonl') for r in []}
if runs:
    d = disagreement(runs)
    print({k: v for k, v in d.items() if k != 'ids'})
    print(len(select_retry_targets(preds, 0.6, d['ids'], ['OCR', 'Number'])), '재추론 대상')

In [ ]:
print(to_markdown(collect(RUNS), ['OCR', 'Number', 'Scene', 'Table']))